In [1]:
!gsutil -m cp -r gs://blockchain_transac_data/jan_subgraph_filtered /content/

Google recommends using Gcloud storage CLI (https://docs.cloud.google.com/storage/docs/discover-object-storage-gcloud) instead of gsutil. Please refer to migration guide (https://docs.cloud.google.com/storage/docs/gsutil-transition-to-gcloud) for assistance.
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000001.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000003.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000005.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000002.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000006.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000009.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000000.parquet...
Copying gs://blockchain_transac_data/jan_subgraph_filtered/part-000000000004.parquet...
Copying gs://blockchain_transac_data/

In [2]:
!ls /content/jan_subgraph_filtered | wc -l

50


In [3]:
!rm -rf /content/spark
!rm -rf spark-3.5.0-bin-hadoop3*

In [4]:
!apt-get update -qq
!apt-get install -y openjdk-17-jdk


W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following additional packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  libatk1.0-data libatspi2.0-0 libgail-common libgail18 libgtk2.0-0
  libgtk2.0-bin libgtk2.0-common librsvg2-common libxcomposite1 libxt-dev
  libxtst6 libxxf86dga1 openjdk-17-jre session-migration x11-utils
Suggested packages:
  gvfs libxt-doc openjdk-17-demo openjdk-17-source visualvm mesa-utils
The following NEW packages will be installed:
  at-spi2-core fonts-dejavu-core fonts-dejavu-extra gsettings-desktop-schemas
  libatk-bridge2.0-0 libatk-wrapper-java libatk-wrapper-java-jni libatk1.0-0
  lib

In [5]:
!wget -q https://archive.apache.org/dist/spark/spark-3.5.0/spark-3.5.0-bin-hadoop3.tgz

In [6]:
!ls -lh

total 382M
drwxr-xr-x 2 root root 4.0K Jul 20 17:17 jan_subgraph_filtered
drwxr-xr-x 1 root root 4.0K Jun  4 13:32 sample_data
-rw-r--r-- 1 root root 382M Sep  9  2023 spark-3.5.0-bin-hadoop3.tgz


In [7]:
!tar -xzf spark-3.5.0-bin-hadoop3.tgz
!mv spark-3.5.0-bin-hadoop3 /content/spark

In [8]:
!ls /content/spark

bin   data	jars	    LICENSE   NOTICE  R		 RELEASE  yarn
conf  examples	kubernetes  licenses  python  README.md  sbin


In [9]:
import os

os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
os.environ["SPARK_HOME"] = "/content/spark"
os.environ["PATH"] += ":/content/spark/bin"

In [10]:
!pip install findspark
!pip install graphframes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 154.7/154.7 kB 1.5 MB/s eta 0:00:00


In [11]:
import findspark
findspark.init("/content/spark")

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("BlockchainGraphAnalysis") \
    .config("spark.driver.memory", "8g") \
    .config("spark.jars.packages", "graphframes:graphframes:0.8.3-spark3.5-s_2.12") \
    .getOrCreate()

spark

In [12]:
df = spark.read.parquet("/content/jan_subgraph_filtered/")

print("Total Rows:", df.count())
df.printSchema()

Total Rows: 4943153
root
 |-- hash: string (nullable = true)
 |-- block_hash: string (nullable = true)
 |-- block_number: long (nullable = true)
 |-- block_timestamp: timestamp_ntz (nullable = true)
 |-- from_address: string (nullable = true)
 |-- to_address: string (nullable = true)
 |-- value: decimal(38,9) (nullable = true)
 |-- gas: long (nullable = true)
 |-- gas_price: long (nullable = true)
 |-- receipt_gas_used: long (nullable = true)
 |-- receipt_status: long (nullable = true)
 |-- nonce: long (nullable = true)
 |-- transaction_index: long (nullable = true)
 |-- input: string (nullable = true)
 |-- transaction_type: long (nullable = true)



In [13]:
from pyspark.sql.functions import col

vertices = (
    df.select(col("from_address").alias("id"))
    .union(df.select(col("to_address").alias("id")))
    .distinct()
)

print("Total unique wallets:", vertices.count())


Total unique wallets: 2361095


In [14]:
edges = df.select(
    col("from_address").alias("src"),
    col("to_address").alias("dst"),
    col("value"),
    col("block_timestamp"),
    col("hash")
)

print("Total edges:", edges.count())


Total edges: 4943153


In [15]:
from graphframes import GraphFrame

g = GraphFrame(vertices, edges)

print("Graph created successfully.")


Graph created successfully.


In [16]:
degrees = g.degrees.cache()

degrees.orderBy("degree", ascending=False).show(20)

+--------------------+------+
|                  id|degree|
+--------------------+------+
|0x28c6c06298d514d...|472680|
|0x6dfc34609a05bc2...|178144|
|0xcffad3200574698...|177829|
|0x974caa59e49682c...|157927|
|0xcad621da75a66c7...|135317|
|0x21a31ee1afc51d9...|112899|
|0x4976a4a02f38326...|112684|
|0xcbd6832ebc203e4...|107042|
|0x56eddb7aa87536c...|105895|
|0x46340b20830761e...|103469|
|0x9696f59e4d72e23...|102674|
|0xdfd5293d8e347df...| 91182|
|0x7a250d5630b4cf5...| 82937|
|0x75e89d5979e4f6f...| 81037|
|0x4675c7e5baafbff...| 69877|
|0xf89d7b9c864f589...| 60065|
|0xdafea492d9c6733...| 58389|
|0x388c818ca8b9251...| 57374|
|0x95222290dd7278a...| 55215|
|0xae45a8240147e61...| 53891|
+--------------------+------+
only showing top 20 rows



In [17]:
g.inDegrees.orderBy("inDegree", ascending=False).show(10)

+--------------------+--------+
|                  id|inDegree|
+--------------------+--------+
|0x28c6c06298d514d...|  384803|
|0xcffad3200574698...|  177805|
|0x6dfc34609a05bc2...|  118345|
|0x974caa59e49682c...|   84916|
|0x7a250d5630b4cf5...|   82937|
|0xcbd6832ebc203e4...|   75422|
|0xcad621da75a66c7...|   75161|
|0x4675c7e5baafbff...|   59240|
|0x388c818ca8b9251...|   57374|
|0x0b95993a39a363d...|   50629|
+--------------------+--------+
only showing top 10 rows



In [18]:
g.outDegrees.orderBy("outDegree", ascending=False).show(10)

+--------------------+---------+
|                  id|outDegree|
+--------------------+---------+
|0x21a31ee1afc51d9...|   112855|
|0x4976a4a02f38326...|   112642|
|0x56eddb7aa87536c...|   105850|
|0x46340b20830761e...|   103420|
|0x9696f59e4d72e23...|   102632|
|0xdfd5293d8e347df...|    91141|
|0x28c6c06298d514d...|    87877|
|0x974caa59e49682c...|    73011|
|0xcad621da75a66c7...|    60156|
|0x6dfc34609a05bc2...|    59799|
+--------------------+---------+
only showing top 10 rows



In [19]:
suspicious = g.degrees.filter("degree > 500 AND degree < 10000")

suspicious.orderBy("degree", ascending=False).show(20)

+--------------------+------+
|                  id|degree|
+--------------------+------+
|0xef1c6e67703c7bd...|  9725|
|0x2db1d8cdf1abe8c...|  9698|
|0x8c8d7c46219d920...|  9545|
|0xcdd37ada79f589c...|  9355|
|0x473780deaf4a2ac...|  8841|
|0x416299aade6443e...|  8723|
|0xfeebabe6b0418ec...|  8389|
|0x1111111254eeb25...|  8330|
|0xbaf6dc2e647aeb6...|  8029|
|0x41b309236c87b1b...|  7758|
|0xef8801eaf234ff8...|  7715|
|0x389044f3ac74720...|  7633|
|0x88dcdd4a0a58b7e...|  7600|
|0xbbd0d4d067d5af2...|  7587|
|0xbfcd86e36d947a9...|  7575|
|0x6596da8b65995d5...|  7529|
|0x77134cbc06cb00b...|  7426|
|0x17f1a51da68d27c...|  7082|
|0xf408bee3443d039...|  7038|
|0xe3d3d0ed702504e...|  6966|
+--------------------+------+
only showing top 20 rows



In [20]:
from pyspark.sql.functions import col

suspicious_splitters = (
    g.outDegrees
    .join(g.inDegrees, "id")
    .filter((col("outDegree") > 500) & (col("inDegree") < 100))
    .orderBy(col("outDegree").desc())
)

suspicious_splitters.show(20)

+--------------------+---------+--------+
|                  id|outDegree|inDegree|
+--------------------+---------+--------+
|0x21a31ee1afc51d9...|   112855|      44|
|0x4976a4a02f38326...|   112642|      42|
|0x56eddb7aa87536c...|   105850|      45|
|0x46340b20830761e...|   103420|      49|
|0x9696f59e4d72e23...|   102632|      42|
|0xdfd5293d8e347df...|    91141|      41|
|0x4d9ff50ef4da947...|    50630|      12|
|0x690b9a9e9aa1c9d...|    47369|       1|
|0x267be1c1d684f78...|    45392|       2|
|0x8216874887415e2...|    41115|      13|
|0x151b381058f91cf...|    40917|      10|
|0x061f7937b7b2bc7...|    35007|       2|
|0xc5a93444cc4da6e...|    33552|       4|
|0x9430801ebaf509a...|    31145|       8|
|0x6081258689a75d2...|    30319|       6|
|0xab97925eb84fe02...|    29718|      88|
|0xcfc0f98f30742b6...|    24470|       3|
|0x19f494583c7c933...|    21758|      29|
|0xd24400ae8bfebb1...|    17420|      22|
|0x74dec05e5b894b0...|    11588|      17|
+--------------------+---------+--

In [21]:
suspicious_collectors = (
    g.inDegrees
    .join(g.outDegrees, "id")
    .filter((col("inDegree") > 500) & (col("outDegree") < 100))
    .orderBy(col("inDegree").desc())
)

suspicious_collectors.show(20)

+--------------------+--------+---------+
|                  id|inDegree|outDegree|
+--------------------+--------+---------+
|0xcffad3200574698...|  177805|       24|
|0xcac0f1a06d3f023...|    5908|        1|
|0x1786d0ca2b790e8...|    5561|       73|
|0xe3792a9c235d434...|    1424|       64|
|0x15f8733104b67f1...|    1394|       19|
|0x3aff86656a65f3d...|    1032|       16|
|0x4de23f3f0fb3318...|     998|       51|
|0x2bfb4dfdefcc4a7...|     942|       14|
|0xc31561478a1e1aa...|     909|       53|
|0xbe83e7db6c6fb4f...|     808|        2|
|0xd1305fbda97c035...|     608|       40|
|0x59141af2478701c...|     540|       10|
|0x3d31a92dc4416b2...|     530|        1|
|0x351c17f198062d7...|     516|       26|
+--------------------+--------+---------+



In [22]:
from pyspark.sql.functions import sum

total_sent = (
    df.groupBy("from_address")
      .agg(sum("value").alias("total_sent"))
)

total_sent.show(10)

+--------------------+--------------------+
|        from_address|          total_sent|
+--------------------+--------------------+
|0x0301aa087ce7365...|77806909963043318...|
|0x0ecbc04e60335b4...|27608532359027000...|
|0x16ef69f9df05774...|14896703507275580...|
|0x27e3aa4727c75e1...|73416172855888244...|
|0x2a7446befa6c0ed...|14999517000000000...|
|0x2a9a6f472073b71...|2819674000000000....|
|0x2b823b348fa2925...|8570778213886927....|
|0x2fcc7c2480cbdbc...|29835952582529000...|
|0x31000adcd6476b0...|14848258440999062...|
|0x33523fa0c3f1b46...|31355730000000000...|
+--------------------+--------------------+
only showing top 10 rows



In [23]:
total_received = (
    df.groupBy("to_address")
      .agg(sum("value").alias("total_received"))
)

total_received.show(10)

+--------------------+--------------------+
|          to_address|      total_received|
+--------------------+--------------------+
|0x23a21218e869674...|1896608445875000....|
|0xdb4cce64806cd5d...|1306908990205000....|
|0xda1472585e91f2b...|1840488605982000....|
|0x3d859479ca379d3...|1304856791040000....|
|0x4d8cb7a76cc21bc...|23536888247212916...|
|0xed42155a89a7495...|14719000000000000...|
|0x9c19b0497997fe9...|54272966024879709...|
|0x1c349c317b07fb4...|15241182062286516...|
|0x3a627f67ef103c4...|13375049660000000...|
|0x7425652a7aeb3b9...|33729040000000000...|
+--------------------+--------------------+
only showing top 10 rows



In [24]:
from pyspark.sql.functions import avg

avg_sent = (
    df.groupBy("from_address")
      .agg(avg("value").alias("avg_sent"))
)

avg_sent.show(10)

+--------------------+--------------------+
|        from_address|            avg_sent|
+--------------------+--------------------+
|0x0301aa087ce7365...|33829091288279703...|
|0x0ecbc04e60335b4...|27608532359027000...|
|0x16ef69f9df05774...|14896703507275580...|
|0x27e3aa4727c75e1...|36708086427944122...|
|0x2a7446befa6c0ed...|14999517000000000...|
|0x2a9a6f472073b71...|2819674000000000....|
|0x2b823b348fa2925...|8570778213886927....|
|0x2fcc7c2480cbdbc...|37294940728161250...|
|0x31000adcd6476b0...|61356439838839100...|
|0x33523fa0c3f1b46...|31355730000000000...|
+--------------------+--------------------+
only showing top 10 rows



In [25]:
avg_received = (
    df.groupBy("to_address")
      .agg(avg("value").alias("avg_received"))
)

avg_received.show(10)

+--------------------+--------------------+
|          to_address|        avg_received|
+--------------------+--------------------+
|0x23a21218e869674...|1896608445875000....|
|0xdb4cce64806cd5d...|1306908990205000....|
|0xda1472585e91f2b...|1840488605982000....|
|0x3d859479ca379d3...|1304856791040000....|
|0x4d8cb7a76cc21bc...|3362412606744702....|
|0xed42155a89a7495...|26283928571428571...|
|0x9c19b0497997fe9...|33543242289789684...|
|0x1c349c317b07fb4...|76205910311432584...|
|0x3a627f67ef103c4...|58152389826086956...|
|0x7425652a7aeb3b9...|16864520000000000...|
+--------------------+--------------------+
only showing top 10 rows



In [26]:
from pyspark.sql.functions import countDistinct

unique_receivers = (
    df.groupBy("from_address")
      .agg(countDistinct("to_address").alias("unique_receivers"))
)

unique_receivers.show(10)

+--------------------+----------------+
|        from_address|unique_receivers|
+--------------------+----------------+
|0x0301aa087ce7365...|               1|
|0x33523fa0c3f1b46...|               1|
|0x9c19b0497997fe9...|            3178|
|0x70d528966c97bd6...|               1|
|0xddc992f0956968c...|               1|
|0x3b2099b1f2bd478...|               1|
|0xcdb1e80a7e8b60f...|               1|
|0x1c7f2219457b0f4...|               1|
|0x987b7ef066f02eb...|               1|
|0x0e093d4f3c013fe...|               1|
+--------------------+----------------+
only showing top 10 rows



In [27]:
unique_senders = (
    df.groupBy("to_address")
      .agg(countDistinct("from_address").alias("unique_senders"))
)

unique_senders.show(10)

+--------------------+--------------+
|          to_address|unique_senders|
+--------------------+--------------+
|0xccc5e935003955d...|             1|
|0x9c19b0497997fe9...|           884|
|0x03705325d69db01...|             1|
|0x01c74e33cff1347...|             1|
|0xae1e1c6394adc5f...|             1|
|0x1b1b85bcf29239c...|             2|
|0x9578b32c74e62fa...|             1|
|0x4cf88fb3e751707...|             1|
|0x88d6e01cefe45f7...|             3|
|0x74bf3d4f4a8f7be...|             1|
+--------------------+--------------+
only showing top 10 rows



In [28]:
from pyspark.sql.functions import to_date

active_days = (
    df.withColumn("day", to_date("block_timestamp"))
      .groupBy("from_address")
      .agg(countDistinct("day").alias("active_days"))
)

active_days.show(10)

+--------------------+-----------+
|        from_address|active_days|
+--------------------+-----------+
|0xa3826cddccf491a...|          1|
|0xb9ea997858e3406...|          3|
|0x12424270bea07ce...|          1|
|0x065cf88c33a751a...|          1|
|0x21d36a469572aea...|          1|
|0xd62e5ce467979b2...|          1|
|0xe577689ff887b1e...|          2|
|0x2023dc702f2b870...|          1|
|0x449c2065754940d...|          1|
|0x34be344c8369d89...|          1|
+--------------------+-----------+
only showing top 10 rows



In [29]:
from pyspark.sql.functions import col

wallet_features = (
    vertices
    .join(degrees, vertices.id == degrees.id, "left")
    .drop(degrees.id)
    .join(g.inDegrees, "id", "left")
    .join(g.outDegrees, "id", "left")
    .join(total_sent.withColumnRenamed("from_address", "id"), "id", "left")
    .join(total_received.withColumnRenamed("to_address", "id"), "id", "left")
    .join(avg_sent.withColumnRenamed("from_address", "id"), "id", "left")
    .join(avg_received.withColumnRenamed("to_address", "id"), "id", "left")
    .join(unique_receivers.withColumnRenamed("from_address", "id"), "id", "left")
    .join(unique_senders.withColumnRenamed("to_address", "id"), "id", "left")
    .join(active_days.withColumnRenamed("from_address", "id"), "id", "left")
)

wallet_features = wallet_features.fillna(0)

wallet_features.show(10)

+--------------------+------+--------+---------+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+
|                  id|degree|inDegree|outDegree|          total_sent|      total_received|            avg_sent|        avg_received|unique_receivers|unique_senders|active_days|
+--------------------+------+--------+---------+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+
|0x0301aa087ce7365...|    23|       0|       23|77806909963043318...|         0.000000000|33829091288279703...|     0.0000000000000|               1|             0|         14|
|0x0ecbc04e60335b4...|    10|       0|       10|27608532359027000...|         0.000000000|27608532359027000...|     0.0000000000000|               1|             0|          8|
|0x16ef69f9df05774...|     1|       0|        1|14896703507275580...|         0.000000000|14896703507275580...|    

In [30]:
from pyspark.sql.functions import when

wallet_features = (
    wallet_features
    .withColumn("fan_out_ratio", col("outDegree") / (col("inDegree") + 1))
    .withColumn("fan_in_ratio", col("inDegree") / (col("outDegree") + 1))
    .withColumn("avg_daily_tx", col("degree") / (col("active_days") + 1))
    .withColumn("total_counterparties",
                col("unique_receivers") + col("unique_senders"))
)

wallet_features.show(10)

+--------------------+------+--------+---------+-----------+--------------------+---------------+--------------------+----------------+--------------+-----------+-------------+------------+------------+--------------------+
|                  id|degree|inDegree|outDegree| total_sent|      total_received|       avg_sent|        avg_received|unique_receivers|unique_senders|active_days|fan_out_ratio|fan_in_ratio|avg_daily_tx|total_counterparties|
+--------------------+------+--------+---------+-----------+--------------------+---------------+--------------------+----------------+--------------+-----------+-------------+------------+------------+--------------------+
|0x000000000464e23...|     1|       1|        0|0.000000000|15130700000000000...|0.0000000000000|15130700000000000...|               0|             1|          0|          0.0|         1.0|         1.0|                   1|
|0x000000006b4bd02...|     1|       1|        0|0.000000000|17291580000000000...|0.0000000000000|1729158

In [31]:
from pyspark.sql.functions import lit

wallet_features = (
    wallet_features
    .withColumn(
        "risk_score",
        lit(0)
        + when(col("degree") > 500, 2).otherwise(0)
        + when(col("degree") > 2000, 3).otherwise(0)
        + when(col("inDegree") > 500, 2).otherwise(0)
        + when(col("outDegree") > 500, 2).otherwise(0)
        + when(col("fan_out_ratio") > 10, 2).otherwise(0)
        + when(col("fan_in_ratio") > 10, 2).otherwise(0)
        + when(col("active_days") > 20, 1).otherwise(0)
    )
)

In [32]:
wallet_features = wallet_features.withColumn(
    "risk_level",
    when(col("risk_score") >= 6, "High")
    .when(col("risk_score") >= 3, "Medium")
    .otherwise("Low")
)

wallet_features.show(20)

+--------------------+------+--------+---------+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+-------------+------------+------------+--------------------+----------+----------+
|                  id|degree|inDegree|outDegree|          total_sent|      total_received|            avg_sent|        avg_received|unique_receivers|unique_senders|active_days|fan_out_ratio|fan_in_ratio|avg_daily_tx|total_counterparties|risk_score|risk_level|
+--------------------+------+--------+---------+--------------------+--------------------+--------------------+--------------------+----------------+--------------+-----------+-------------+------------+------------+--------------------+----------+----------+
|0x000000000464e23...|     1|       1|        0|         0.000000000|15130700000000000...|     0.0000000000000|15130700000000000...|               0|             1|          0|          0.0|         1.0|         1.0|    

In [33]:
wallet_features.orderBy(
    col("risk_score").desc(),
    col("degree").desc()
).show(20, truncate=False)

+------------------------------------------+------+--------+---------+-----------------------------------+-----------------------------------+----------------------------------+-------------------------------------+----------------+--------------+-----------+--------------------+---------------------+-----------------+--------------------+----------+----------+
|id                                        |degree|inDegree|outDegree|total_sent                         |total_received                     |avg_sent                          |avg_received                         |unique_receivers|unique_senders|active_days|fan_out_ratio       |fan_in_ratio         |avg_daily_tx     |total_counterparties|risk_score|risk_level|
+------------------------------------------+------+--------+---------+-----------------------------------+-----------------------------------+----------------------------------+-------------------------------------+----------------+--------------+-----------+-------------

In [34]:
wallet_features.orderBy(
    col("total_sent").desc()
).select(
    "id",
    "total_sent",
    "outDegree",
    "risk_score",
    "risk_level"
).show(20, truncate=False)

+------------------------------------------+-----------------------------------+---------+----------+----------+
|id                                        |total_sent                         |outDegree|risk_score|risk_level|
+------------------------------------------+-----------------------------------+---------+----------+----------+
|0x28c6c06298d514db089934071355e5743bf21d60|4982679343300900000000000.000000000|87877    |10        |High      |
|0x267be1c1d684f78cb4f6a176c4911b741e4ffdc0|822026406158412700000000.000000000 |45392    |10        |High      |
|0xb3f923eabaf178fc1bd8e13902fc5c61d3ddef5b|672677946004944341557820.000000000 |444      |3         |Medium    |
|0xf584f8728b874a6a5c7a8d4d387c9aae9172d621|652661803903000000000000.000000000 |1800     |10        |High      |
|0x21a31ee1afc51d94c2efccaa2092ad1028285549|526387704441610000000000.000000000 |112855   |10        |High      |
|0x56eddb7aa87536c09ccc2793473599fd21a8b17f|511269567714650000000000.000000000 |105850   |10    

In [35]:
wallet_features.orderBy(
    col("total_received").desc()
).select(
    "id",
    "total_received",
    "inDegree",
    "risk_score",
    "risk_level"
).show(20, truncate=False)

+------------------------------------------+-----------------------------------+--------+----------+----------+
|id                                        |total_received                     |inDegree|risk_score|risk_level|
+------------------------------------------+-----------------------------------+--------+----------+----------+
|0x28c6c06298d514db089934071355e5743bf21d60|4073691905495216272212251.000000000|384803  |10        |High      |
|0x21a31ee1afc51d94c2efccaa2092ad1028285549|934699051164851299186911.000000000 |44      |10        |High      |
|0x4976a4a02f38326660d17bf34b431dc6e2eb2327|922887767988978527433769.000000000 |42      |10        |High      |
|0x56eddb7aa87536c09ccc2793473599fd21a8b17f|911709996457812807841293.000000000 |45      |10        |High      |
|0xdfd5293d8e347dfe59e90efd55b2956a1343963d|805854020950360905082144.000000000 |41      |10        |High      |
|0x9696f59e4d72e237be84ffd425dcad154bf96976|804387977569244647063181.000000000 |42      |10        |High

In [36]:
wallet_features.groupBy("risk_level").count().show()

+----------+-------+
|risk_level|  count|
+----------+-------+
|      High|    605|
|       Low|2359655|
|    Medium|    835|
+----------+-------+



In [37]:
wallet_features.coalesce(1).write.mode("overwrite").option(
    "header", True
).csv("/content/final_wallet_features_csv")

In [38]:
# wallet_features.write.mode("overwrite").parquet(
#     "/content/final_wallet_features_parquet"
# )

In [39]:
print("Total Wallets :", wallet_features.count())

print("High Risk :", wallet_features.filter(col("risk_level")=="High").count())

print("Medium Risk :", wallet_features.filter(col("risk_level")=="Medium").count())

print("Low Risk :", wallet_features.filter(col("risk_level")=="Low").count())

Total Wallets : 2361095
High Risk : 605
Medium Risk : 835
Low Risk : 2359655
